# Notebook 1: Hallucination detection without context (baseline)

This notebook runs two small language models (Qwen3-4B and Phi-4-mini) on ARC-Challenge science questions **using only the question, with no retrieved passages**. For every question the model gives one main answer and 5 sampled answers. Two detectors, D0 (answer agreement) and D1 (explanation similarity), are then evaluated on how well they separate wrong answers from correct ones. These results are the baseline for the with-context experiment in Notebook 3.

## 1. Install packages
Installs the libraries needed for data handling, loading models and datasets.

In [ ]:

!pip -q install pandas==2.2.3 transformers accelerate datasets

## 2. Imports, Hugging Face login and Google Drive
Imports the main libraries, logs in to Hugging Face with the token stored in Colab Secrets (`HF_TOKEN`), mounts Google Drive and creates the project folder `hallu_project` where all results are saved. It also prints whether a GPU is available.

In [ ]:
import json, os, re, random, time, gc, platform
from pathlib import Path
import numpy as np, pandas as pd, torch, transformers
from google.colab import drive, userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))
drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/hallu_project")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.is_available(), "|", sorted(p.name for p in PROJECT_DIR.glob("*")))

Mounted at /content/drive
GPU: False | ['gen_phi4_mini_no_ctx_dev.jsonl', 'gen_phi4_mini_no_ctx_test.jsonl', 'gen_qwen3_4b_no_ctx_dev.jsonl', 'gen_qwen3_4b_no_ctx_test.jsonl', 'pilot_phi4_mini.csv', 'pilot_qwen3_4b.csv', 'question_sets.json', 'run_info.jsonl']


## 3. Load ARC-Challenge and create fixed question sets
Loads the ARC-Challenge dataset and defines how a question is formatted (options A to E) and how the answer letter is read from the model output. It randomly picks 150 dev questions (validation split) and 300 test questions (test split) with seed 42, saves their ids to `question_sets.json` so every notebook uses the same questions, and records the environment details in `run_info.jsonl`.

In [ ]:
from datasets import load_dataset
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge")

def format_q(ex):
    labels, texts = ex["choices"]["label"], ex["choices"]["text"]
    letters = "ABCDE"[:len(labels)]
    opts = "\n".join(f"{l}. {t}" for l, t in zip(letters, texts))
    gold = letters[labels.index(ex["answerKey"])]
    return ex["question"], opts, gold, letters

def parse_letter(text, letters):
    m = re.search(r"Answer:\s*\(?([A-E])\b", text)
    return m.group(1) if m and m.group(1) in letters else None

sets_path = PROJECT_DIR / "question_sets.json"
if sets_path.exists():
    sets = json.load(open(sets_path))
else:
    rng = random.Random(42)
    dev_idx = rng.sample(range(len(arc["validation"])), 150)
    test_idx = rng.sample(range(len(arc["test"])), 300)
    sets = {"dev": [arc["validation"][i]["id"] for i in dev_idx],
            "test": [arc["test"][i]["id"] for i in test_idx]}
    json.dump(sets, open(sets_path, "w"))
by_id = {ex["id"]: ex for s in ("validation", "test") for ex in arc[s]}
print("dev:", len(sets["dev"]), "| test:", len(sets["test"]))

info = dict(date=time.strftime("%Y-%m-%d %H:%M"), python=platform.python_version(),
            torch=torch.__version__, transformers=transformers.__version__,
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
with open(PROJECT_DIR / "run_info.jsonl", "a") as f:
    f.write(json.dumps(info) + "\n")
print(info)

README.md:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B /  190kB            

ARC-Challenge/train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

ARC-Challenge/test-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B /  204kB            

ARC-Challenge/test-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

ARC-Challenge/validation-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 55.7kB            

ARC-Challenge/validation-00000-of-00001.(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

dev: 150 | test: 300
{'date': '2026-10-28 09:16', 'python': '3.13.16', 'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'gpu': 'Tesla T4'}


## 4. Model and generation helpers
Defines how to load a model, build the prompt, get one main answer (greedy decoding) and 5 sampled answers (temperature 0.7). `run_condition` goes through a whole split, saves every result to a `.jsonl` file in Drive and skips questions that are already done, so an interrupted run can simply be started again. In this notebook the condition is `no_ctx`, which means no context is added to the prompt.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map="auto")
    return tok, model

def build_prompt(question, opts, context=None):
    ctx = f"Context:\n{context}\n\n" if context else ""
    return ("Answer the science question. Reply in exactly this format:\n"
            "Answer: <option letter>\nReason: <one sentence>\n\n"
            f"{ctx}Question: {question}\n{opts}")

def _inputs(tok, model, question, opts, context):
    text = tok.apply_chat_template(
        [{"role": "user", "content": build_prompt(question, opts, context)}],
        tokenize=False, add_generation_prompt=True)
    return tok(text, return_tensors="pt").to(model.device)

def _decode(tok, out, n_in):
    return [tok.decode(o[n_in:], skip_special_tokens=True) for o in out]

def ask(tok, model, question, opts, context=None, max_new_tokens=60):
    inp = _inputs(tok, model, question, opts, context)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return _decode(tok, out, inp["input_ids"].shape[1])[0]

def ask_many(tok, model, question, opts, context=None, n=5, temperature=0.7, max_new_tokens=60):
    inp = _inputs(tok, model, question, opts, context)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=temperature, top_p=1.0, num_return_sequences=n)
    return _decode(tok, out, inp["input_ids"].shape[1])

def run_condition(tok, model, tag, split, condition="no_ctx", contexts=None):
    path = PROJECT_DIR / f"gen_{tag}_{condition}_{split}.jsonl"
    done = set()
    if path.exists():
        with open(path) as f:
            done = {json.loads(l)["id"] for l in f}
    ids = [i for i in sets[split] if i not in done]
    print(f"{tag} {condition} {split}: {len(done)} done, {len(ids)} to go")
    t_start = time.time()
    with open(path, "a") as f:
        for k, qid in enumerate(ids, 1):
            ex = by_id[qid]
            q, opts, gold, letters = format_q(ex)
            ctx = contexts.get(qid) if contexts else None
            t0 = time.time(); main = ask(tok, model, q, opts, ctx); t1 = time.time()
            samples = ask_many(tok, model, q, opts, ctx); t2 = time.time()
            pred = parse_letter(main, letters)
            rec = dict(id=qid, split=split, condition=condition, model=tag, gold=gold,
                       main=main, pred=pred, correct=(pred == gold),
                       samples=samples, sample_preds=[parse_letter(s, letters) for s in samples],
                       sec_main=round(t1 - t0, 2), sec_samples=round(t2 - t1, 2))
            f.write(json.dumps(rec) + "\n"); f.flush()
            if k % 25 == 0:
                print(f"  {k}/{len(ids)} | {round((time.time()-t_start)/60,1)} min")
    print("finished", path.name)

## 5. Analysis helpers (D0 and AUROC)
Functions to load saved generations, compute answer agreement (the basis of D0), calculate bootstrap confidence intervals for AUROC, and print a summary of accuracy, answer consistency and D0 AUROC. D0 score = 1 minus the share of the 5 samples that match the main answer.

In [ ]:
from sklearn.metrics import roc_auc_score

def load_gen(tag, condition, split):
    path = PROJECT_DIR / f"gen_{tag}_{condition}_{split}.jsonl"
    return pd.DataFrame([json.loads(l) for l in open(path)])

def agreement(r):
    return np.mean([p == r.pred for p in r.sample_preds])

def boot_auc(y, s, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    y, s = np.asarray(y), np.asarray(s)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        vals.append(roc_auc_score(y[idx], s[idx]))
    return np.percentile(vals, [2.5, 97.5])

def summarize(tag, condition, split):
    d = load_gen(tag, condition, split)
    err = (~d.correct).values
    score = 1 - d.apply(agreement, axis=1).values
    wrong = d[err]
    print(f"{tag} | {condition} | {split}: n={len(d)}, errors={err.sum()}, accuracy={d.correct.mean():.3f}")
    print("  main unparsed:", int(d.pred.isna().sum()),
          "| all-5-agree:", round(d.sample_preds.apply(lambda ps: len(set(ps)) == 1).mean(), 3))
    print("  wrong & all 5 repeat same wrong option:",
          round(wrong.apply(lambda r: all(p == r.pred for p in r.sample_preds), axis=1).mean(), 3))
    auc = roc_auc_score(err, score); lo, hi = boot_auc(err, score)
    print(f"  D0 AUROC: {auc:.3f}  (95% CI {lo:.3f} to {hi:.3f})")
    return d

## 6. Load a model and generate on the dev set
Chooses the model with the `tag` line (change only that line to switch between `qwen3_4b` and `phi4_mini`), frees the previous model from GPU memory, loads the new model and generates answers for the 150 dev questions without context.

In [ ]:
MODELS = {"qwen3_4b":  "Qwen/Qwen3-4B-Instruct-2507",
          "phi4_mini": "microsoft/Phi-4-mini-instruct"}

tag = "phi4_mini"        # <- For changing model, change this line only ("qwen3_4b" ya "phi4_mini")

if "model" in globals():                     # remove previous model from memory
    del model; gc.collect(); torch.cuda.empty_cache()

tok, model = load_model(MODELS[tag])
run_condition(tok, model, tag, "dev")

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

phi4_mini no_ctx dev: 150 done, 0 to go
finished gen_phi4_mini_no_ctx_dev.jsonl


## 7. Generate on the test set
Uses the model loaded above to generate answers for the 300 test questions without context. Run cells 6 and 7 once for each model.

In [ ]:
run_condition(tok, model, tag, "test")

phi4_mini no_ctx test: 300 done, 0 to go
finished gen_phi4_mini_no_ctx_test.jsonl


## 8. D0 summary for both models
Prints accuracy, number of errors, how often all 5 samples agree, how often a wrong answer is repeated in all 5 samples, and D0 AUROC (with 95% confidence interval) for both models on dev and test.

In [ ]:
for t in ["qwen3_4b", "phi4_mini"]:
    for s in ["dev", "test"]:
        summarize(t, "no_ctx", s)

qwen3_4b | no_ctx | dev: n=150, errors=15, accuracy=0.900
  main unparsed: 0 | all-5-agree: 0.993
  wrong & all 5 repeat same wrong option: 0.933
  D0 AUROC: 0.533  (95% CI 0.500 to 0.611)
qwen3_4b | no_ctx | test: n=300, errors=32, accuracy=0.893
  main unparsed: 0 | all-5-agree: 0.99
  wrong & all 5 repeat same wrong option: 0.938
  D0 AUROC: 0.530  (95% CI 0.496 to 0.579)
phi4_mini | no_ctx | dev: n=150, errors=25, accuracy=0.833
  main unparsed: 0 | all-5-agree: 0.747
  wrong & all 5 repeat same wrong option: 0.32
  D0 AUROC: 0.741  (95% CI 0.639 to 0.835)
phi4_mini | no_ctx | test: n=300, errors=54, accuracy=0.820
  main unparsed: 0 | all-5-agree: 0.817
  wrong & all 5 repeat same wrong option: 0.593
  D0 AUROC: 0.642  (95% CI 0.571 to 0.706)


## 9. Load the sentence embedding model
Loads all-MiniLM-L6-v2, which turns each explanation into a vector so that explanations can be compared. It is used for the D1 detector. The package is installed first if it is missing.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    !pip -q install sentence-transformers
    from sentence_transformers import SentenceTransformer

emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("embedding model ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding model ready


## 10. D1 detector and evaluation function
Extracts the one-sentence reason from each answer, measures how similar the main reason is to the 5 sampled reasons, and defines D1 as 1 minus the average similarity. `eval_d1` computes D0 and D1 AUROC with confidence intervals, shows the D1 score for correct and wrong answers, and saves the scores to a CSV file in Drive.

In [ ]:
def reason(txt):
    m = re.search(r"Reason:\s*(.*)", txt, re.S)
    return (m.group(1) if m else txt).strip()

def d1_score(r):
    texts = [reason(r.main)] + [reason(s) for s in r.samples]
    E = emb.encode(texts, normalize_embeddings=True)
    return 1 - float(np.mean(E[1:] @ E[0]))

def eval_d1(tag, condition, split):
    d = load_gen(tag, condition, split)
    err = (~d.correct).values
    d0 = 1 - d.apply(agreement, axis=1).values
    d1 = d.apply(d1_score, axis=1).values
    out = pd.DataFrame({"id": d.id, "error": err, "d0": d0, "d1": d1})
    out.to_csv(PROJECT_DIR / f"scores_{tag}_{condition}_{split}.csv", index=False)
    a0, a1 = roc_auc_score(err, d0), roc_auc_score(err, d1)
    l0, h0 = boot_auc(err, d0); l1, h1 = boot_auc(err, d1)
    print(f"{tag} | {condition} | {split}: n={len(d)}, errors={err.sum()}")
    print(f"  D0 AUROC {a0:.3f} (95% CI {l0:.3f}-{h0:.3f}) | D1 AUROC {a1:.3f} (95% CI {l1:.3f}-{h1:.3f})")
    print(f"  D1 mean score: correct={d1[~err].mean():.4f} | wrong={d1[err].mean():.4f}")
    silent = (d0 == 0)
    if err[silent].sum() > 0 and (~err[silent]).sum() > 0:
        print(f"  where D0 is silent (all 5 agree, n={silent.sum()}): errors={err[silent].sum()}, D1 AUROC {roc_auc_score(err[silent], d1[silent]):.3f}")
    return out

## 11. D0 and D1 results on the dev set
Computes D0 and D1 AUROC for the dev set of both models and saves the scores to `scores_<model>_no_ctx_dev.csv`.

In [ ]:
for t in ["qwen3_4b", "phi4_mini"]:
    eval_d1(t, "no_ctx", "dev")

qwen3_4b | no_ctx | dev: n=150, errors=15
  D0 AUROC 0.533 (95% CI 0.500-0.611) | D1 AUROC 0.672 (95% CI 0.527-0.795)
  D1 mean score: correct=0.0418 | wrong=0.0652
  where D0 is silent (all 5 agree, n=149): errors=14, D1 AUROC 0.651
phi4_mini | no_ctx | dev: n=150, errors=25
  D0 AUROC 0.741 (95% CI 0.639-0.835) | D1 AUROC 0.689 (95% CI 0.581-0.793)
  D1 mean score: correct=0.1113 | wrong=0.1612
  where D0 is silent (all 5 agree, n=112): errors=8, D1 AUROC 0.593


## 12. D0 and D1 results on the test set
Computes D0 and D1 AUROC for the test set of both models and saves the scores to `scores_<model>_no_ctx_test.csv`. These test results are the baseline used in the final comparison.

In [ ]:
for t in ["qwen3_4b", "phi4_mini"]:
    eval_d1(t, "no_ctx", "test")

qwen3_4b | no_ctx | test: n=300, errors=32
  D0 AUROC 0.530 (95% CI 0.496-0.579) | D1 AUROC 0.594 (95% CI 0.483-0.704)
  D1 mean score: correct=0.0467 | wrong=0.0671
  where D0 is silent (all 5 agree, n=297): errors=30, D1 AUROC 0.575
phi4_mini | no_ctx | test: n=300, errors=54
  D0 AUROC 0.642 (95% CI 0.571-0.706) | D1 AUROC 0.655 (95% CI 0.582-0.724)
  D1 mean score: correct=0.1104 | wrong=0.1587
  where D0 is silent (all 5 agree, n=245): errors=32, D1 AUROC 0.640


## 13. Paired difference between D1 and D0
Uses bootstrap resampling on the test set to check whether the AUROC difference between D1 and D0 is real or could be noise. If the 95% confidence interval includes 0, the difference is not statistically clear.

In [ ]:
def paired_diff(tag, split="test", n=2000, seed=0):
    s = pd.read_csv(PROJECT_DIR / f"scores_{tag}_no_ctx_{split}.csv")
    y, d0, d1 = s.error.values.astype(int), s.d0.values, s.d1.values
    rng = np.random.default_rng(seed); diffs = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        diffs.append(roc_auc_score(y[i], d1[i]) - roc_auc_score(y[i], d0[i]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"{tag} {split}: D1-D0 AUROC diff = {roc_auc_score(y, d1) - roc_auc_score(y, d0):+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")

for t in ["qwen3_4b", "phi4_mini"]:
    paired_diff(t)

qwen3_4b test: D1-D0 AUROC diff = +0.065 (95% CI -0.046 to +0.174)
phi4_mini test: D1-D0 AUROC diff = +0.013 (95% CI -0.075 to +0.096)
